In [1]:
from git import Repo
from langchain_classic.text_splitter import Language, RecursiveCharacterTextSplitter
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers.language.language_parser import LanguageParser
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.chat_models.openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_classic.chains import ConversationalRetrievalChain
import os


C:\Users\HP\AppData\Local\Temp\ipykernel_23928\3381987435.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.generic import GenericLoader


In [2]:
!mkdir -p test_repo

In [3]:
repo_path = 'test_repo'
repo = Repo.clone_from("https://github.com/entbappy/End-to-end-Medical-Chatbot-Generative-AI", to_path=repo_path )

In [4]:
%pwd

'e:\\Kirsh Naik Academy\\Code Reviewer\\RealTimeSourceCodeAnalyzer\\research'

In [5]:
loader = GenericLoader.from_filesystem(repo_path,
                                        glob = "**/*",
                                       suffixes=[".py"],
                                       parser = LanguageParser(language=Language.PYTHON, parser_threshold=500)
)

In [6]:
documents = loader.load()

In [7]:
documents

[Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='from flask import Flask, render_template, jsonify, request\nfrom src.helper import download_hugging_face_embeddings\nfrom langchain_pinecone import PineconeVectorStore\nfrom langchain_openai import OpenAI\nfrom langchain.chains import create_retrieval_chain\nfrom langchain.chains.combine_documents import create_stuff_documents_chain\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom dotenv import load_dotenv\nfrom src.prompt import *\nimport os\n\napp = Flask(__name__)\n\nload_dotenv()\n\nPINECONE_API_KEY=os.environ.get(\'PINECONE_API_KEY\')\nOPENAI_API_KEY=os.environ.get(\'OPENAI_API_KEY\')\n\nos.environ["PINECONE_API_KEY"] = PINECONE_API_KEY\nos.environ["OPENAI_API_KEY"] = OPENAI_API_KEY\n\nembeddings = download_hugging_face_embeddings()\n\n\nindex_name = "medicalbot"\n\n# Embed each chunk and upsert the embeddings into your Pinecone index.\ndocsearch = PineconeVectorS

In [8]:
len(documents)

7

In [9]:
documents_splitter = RecursiveCharacterTextSplitter.from_language(language = Language.PYTHON,
                                                             chunk_size = 500,
                                                             chunk_overlap = 20)

In [10]:
texts = documents_splitter.split_documents(documents)

In [14]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY=os.environ.get('OPENAI_API_KEY')

In [15]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [16]:
embeddings=OpenAIEmbeddings(disallowed_special=())

C:\Users\HP\AppData\Local\Temp\ipykernel_23928\972742285.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import OpenAIEmbeddings``.
  embeddings=OpenAIEmbeddings(disallowed_special=())


In [17]:

vectordb = Chroma.from_documents(texts, embedding=embeddings, persist_directory='./data')

In [18]:
vectordb.persist()

C:\Users\HP\AppData\Local\Temp\ipykernel_23928\3711397106.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [19]:
llm = ChatOpenAI()

C:\Users\HP\AppData\Local\Temp\ipykernel_23928\44520985.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI()


In [20]:
memory = ConversationSummaryMemory(llm=llm, memory_key = "chat_history", return_messages=True)

C:\Users\HP\AppData\Local\Temp\ipykernel_23928\2101274949.py:1: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(llm=llm, memory_key = "chat_history", return_messages=True)


In [21]:
qa = ConversationalRetrievalChain.from_llm(llm, retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k":8}), memory=memory)

In [22]:
question = "what is download_hugging_face_embeddings function?"

In [23]:
result = qa(question)

C:\Users\HP\AppData\Local\Temp\ipykernel_23928\2910481472.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(question)


In [24]:
print(result)

{'question': 'what is download_hugging_face_embeddings function?', 'chat_history': [SystemMessage(content='', additional_kwargs={}, response_metadata={})], 'answer': 'The `download_hugging_face_embeddings` function is a function that downloads embeddings from the Hugging Face model called "sentence-transformers/all-MiniLM-L6-v2." This model returns embeddings with 384 dimensions, which are used for natural language processing tasks like text similarity and clustering.'}
